# Lab - kích hoạt ReLU


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
plt.style.use('./deeplearning.mplstyle')
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU
from tensorflow.keras.activations import linear, relu, sigmoid
%matplotlib widget
from matplotlib.widgets import Slider
from lab_utils_common import dlc
from autils import plt_act_trio
from lab_utils_relu import *
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)


<a name="2"></a>
##2 - ReLU Kích hoạt
Tuần này, một kích hoạt mới đã được giới thiệu, Đơn vị tuyến tính chỉnh lưu (ReLU). 
$$ a = max(0,z) \quad\quad\text {# ReLU function} $$


In [2]:
plt_act_trio()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

<img align="right" src="./images/C2_W2_ReLu.png"     style=" width:380px; padding: 10px 20px; " >
Ví dụ từ bài giảng bên phải cho thấy ứng dụng của ReLU. Trong ví dụ này, feature "nhận thức" dẫn xuất không phải là nhị phân mà có phạm vi giá trị liên tục. sigmoid là tốt nhất cho các tình huống bật/tắt hoặc nhị phân. ReLU cung cấp mối quan hệ tuyến tính liên tục. Ngoài ra, nó có phạm vi 'tắt' trong đó đầu ra bằng 0.     
feature "tắt" làm cho ReLU kích hoạt Phi tuyến tính. Tại sao điều này lại cần thiết? Hãy xem xét điều này dưới đây.


### Tại sao kích hoạt phi tuyến tính?  
<img align="left" src="./images/C2_W2_ReLU_Graph.png"     style=" width:250px; padding: 10px 20px; " > Hàm hiển thị bao gồm các phần tuyến tính (tuyến tính từng phần). Độ dốc nhất quán trong phần tuyến tính và sau đó thay đổi đột ngột tại các điểm chuyển tiếp. Tại các điểm chuyển tiếp, một hàm tuyến tính mới được thêm vào, khi được thêm vào hàm hiện có sẽ tạo ra độ dốc mới. Hàm mới được thêm vào tại điểm chuyển tiếp nhưng không đóng góp vào đầu ra trước thời điểm đó. Hàm kích hoạt phi tuyến tính chịu trách nhiệm vô hiệu hóa đầu vào trước và đôi khi sau các điểm chuyển tiếp. Bài tập sau đây cung cấp một ví dụ rõ ràng hơn.


Bài tập sẽ sử dụng mạng bên dưới trong một bài toán hồi quy trong đó bạn phải lập mô hình target tuyến tính từng phần:
<img align="center" src="./images/C2_W2_ReLU_Network.png"     style=" width:650px; padding: 10px 20px; ">  
Mạng có 3 đơn vị ở lớp đầu tiên. Mỗi cái đều được yêu cầu để hình thành target. Đơn vị 0 được lập trình sẵn và cố định để ánh xạ đoạn đầu tiên. Bạn sẽ sửa đổi trọng số và độ lệch trong phần 1 và 2 để lập mô hình phân đoạn thứ 2 và thứ 3. Đơn vị đầu ra cũng được cố định và chỉ tính tổng các đầu ra của lớp đầu tiên.  

Sử dụng thanh trượt bên dưới, sửa đổi trọng số và độ lệch để phù hợp với target. 
Gợi ý: Bắt đầu bằng `w1` và `b1` rồi để lại `w2` và `b2` 0 cho đến khi bạn khớp với phân đoạn thứ 2. Nhấp chuột thay vì trượt nhanh hơn.  Nếu bạn gặp khó khăn, đừng lo lắng, phần văn bản bên dưới sẽ mô tả điều này chi tiết hơn.


In [3]:
_ = plt_relu_ex()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

target của bài tập này là đánh giá cao hành vi phi tuyến tính của ReLU cung cấp khả năng cần thiết để tắt các chức năng cho đến khi chúng cần thiết. Hãy xem điều này hoạt động như thế nào trong ví dụ này.
<img align="right" src="./images/C2_W2_ReLU_Plot.png"     style=" width:600px; padding: 10px 20px; "> 
Các ô bên phải chứa đầu ra của các đơn vị ở lớp đầu tiên.   
Bắt đầu từ trên cùng, đơn vị 0 chịu trách nhiệm về phân đoạn đầu tiên được đánh dấu bằng 1. Cả hàm tuyến tính $z$ và hàm theo sau ReLU $a$ đều được hiển thị. Bạn có thể thấy rằng ReLU sẽ tắt chức năng sau khoảng [0,1]. Điều này rất quan trọng vì nó ngăn Đơn vị 0 can thiệp vào phân đoạn sau. 

Đơn vị 1 chịu trách nhiệm về phân đoạn thứ 2. Ở đây, ReLU giữ đơn vị này ở yên cho đến sau x là 1. Vì đơn vị đầu tiên không đóng góp nên độ dốc của đơn vị 1, $w^{[1]}_1$, chỉ là độ dốc của đường target. Độ lệch phải được điều chỉnh để giữ đầu ra âm cho đến khi x đạt 1. Lưu ý cách đóng góp của Đơn vị 1 cũng mở rộng sang phân đoạn thứ 3.

Đơn vị 2 chịu trách nhiệm về phân đoạn thứ 3. ReLU lại đưa đầu ra về 0 cho đến khi x đạt giá trị phù hợp. Độ dốc của đơn vị, $w^{[1]}_2$, phải được đặt sao cho tổng của đơn vị 1 và 2 có độ dốc mong muốn. Độ lệch một lần nữa được điều chỉnh để giữ đầu ra âm cho đến khi x đạt 2. 

feature "tắt" hoặc tắt của kích hoạt ReLU cho phép các mô hình ghép các đoạn tuyến tính lại với nhau để mô hình hóa các hàm phi tuyến tính phức tạp.


## Chúc mừng!
Bây giờ bạn đã quen thuộc hơn với ReLU và tầm quan trọng của hành vi phi tuyến tính của nó.
